# 📚 SQL Ch.7 — Window Functions
> BigQuery SQL Reference Guide, Chapter 7: OVER/PARTITION BY · ROW_NUMBER/RANK/DENSE_RANK · LAG/LEAD · SUM OVER · FIRST_VALUE/LAST_VALUE · NTILE · QUALIFY  
> BigQuery SQL 완전 참조 가이드 7장: OVER/PARTITION BY · ROW_NUMBER/RANK/DENSE_RANK · LAG/LEAD · SUM OVER · FIRST_VALUE/LAST_VALUE · NTILE · QUALIFY

---
# 🎯 Learning Objective
Today I want to learn: / 오늘 배우고 싶은 것:
- [x] Explain the one key difference between `GROUP BY` and a window function — row count — and why that makes window functions strictly more flexible  
`GROUP BY`와 윈도우 함수의 핵심 차이(행 수)를 설명하고, 그 차이가 왜 윈도우 함수를 더 유연하게 만드는지 이해한다
- [x] Rank rows with `ROW_NUMBER`/`RANK`/`DENSE_RANK` and pick the right one based on how ties should behave  
`ROW_NUMBER`/`RANK`/`DENSE_RANK`로 순위를 매기고, 동점 처리 방식에 따라 알맞은 것을 고른다
- [x] Compare each row to the previous one with `LAG`, and combine `RANK`/`QUALIFY` to pull a top-N per group in one query  
`LAG`로 각 행을 이전 행과 비교하고, `RANK`/`QUALIFY`를 조합해 그룹별 상위 N개를 한 쿼리로 뽑는다

---
# 🧠 Concept

## What is it?
*(Explain it in your own words.)*

**EN:** A window function calculates something across a group of related rows — a sum, a rank, the previous row's value — **without collapsing those rows into one**, unlike `GROUP BY`. Every original row survives; the window function just attaches a calculated value onto each one, computed over a "window" of rows you define with `PARTITION BY` (which rows belong together) and `ORDER BY` (what order to consider them in).

**KR:** 윈도우 함수는 관련된 행들의 그룹에 걸쳐 무언가를 계산합니다 — 합계, 순위, 이전 행의 값 등 — 하지만 `GROUP BY`와 달리 **그 행들을 하나로 합치지 않습니다**. 원본 행이 모두 그대로 살아남고, 윈도우 함수는 `PARTITION BY`(어느 행들이 한 그룹인지)와 `ORDER BY`(어떤 순서로 볼지)로 정의한 "창(window)" 위에서 계산한 값을 각 행에 붙일 뿐입니다.

## Why do we use it?
*(When is it useful?)*

**EN:** Some questions genuinely need both the group-level number *and* the individual row, side by side — "this order's amount, next to its region's total" — and `GROUP BY` alone physically can't produce that (grouping is what erases the individual rows). Window functions are also the only clean way to ask relative questions: "what's the rank of this row," "what was the previous row's value," "what's the running total up to this point."

**KR:** 어떤 질문들은 그룹 단위 숫자*와* 개별 행을 나란히 필요로 합니다 — "이 주문의 금액, 그 옆에 해당 지역의 총합" — `GROUP BY`만으로는 물리적으로 이걸 만들 수 없습니다(그룹화 자체가 개별 행을 지우는 일이기 때문). 윈도우 함수는 또한 상대적인 질문을 하는 유일하게 깔끔한 방법입니다: "이 행의 순위는", "이전 행의 값은 뭐였나", "여기까지의 누적 합계는".

## When is it used in Business Analytics?
*(Real-world use case)*

**EN:** "Top 3 products per category," "this month vs. last month," "running total year-to-date," "each customer's most recent order" — this is the toolkit behind almost every dashboard chart that isn't a flat table. Window functions are what separate a basic report ("total by group") from an analytical one ("rank, trend, and comparison within each group").

**KR:** "카테고리별 상위 3개 제품", "이번 달 vs 지난달", "연초부터의 누적 합계", "각 고객의 가장 최근 주문" — 단순 표가 아닌 거의 모든 대시보드 차트 뒤에 이 도구들이 있습니다. 윈도우 함수는 기본 리포트("그룹별 합계")와 분석적인 리포트("그룹 내 순위, 추세, 비교")를 가르는 지점입니다.

**Comparison / 비교표:**

| Task / 작업 | SQL | Pandas |
|---|---|---|
| Group but keep every row / 그룹화하되 모든 행 유지 | window function `OVER (PARTITION BY ...)` | `.groupby().transform()` |
| Collapse to one row per group / 그룹당 한 행으로 압축 | `GROUP BY` | `.groupby().agg()` |
| Rank rows / 순위 매기기 | `ROW_NUMBER`/`RANK`/`DENSE_RANK` | `.rank()` |
| Compare to previous row / 이전 행과 비교 | `LAG`/`LEAD` | `.shift()` |
| Running total / 누적 합계 | `SUM() OVER (... ROWS BETWEEN ...)` | `.cumsum()` |
| Moving average / 이동 평균 | `AVG() OVER (... ROWS BETWEEN ...)` | `.rolling().mean()` |
| Split into equal buckets / 균등 버킷 나누기 | `NTILE` | `pd.qcut()` |
| Filter on a window result / 윈도우 결과 필터링 | `QUALIFY` | filter after `.transform()` |

---
# 📝 Syntax

## Basic Syntax
The core difference: `GROUP BY` collapses rows; a window function (`OVER (...)`) keeps every row and attaches a calculated value to each one.  
핵심 차이: `GROUP BY`는 행을 압축하고, 윈도우 함수(`OVER (...)`)는 모든 행을 유지하면서 계산된 값을 각 행에 붙입니다.

In [1]:
# --- Environment setup / 환경 설정 ---
# We use DuckDB: a free, in-memory SQL engine that understands BigQuery-style syntax
# almost 1:1 (window functions, QUALIFY, ROLLUP, STRING_AGG, etc.), and can query
# pandas DataFrames directly by name -- no separate "load data" step needed.
# DuckDB는 무료 인메모리 SQL 엔진으로, BigQuery 문법(윈도우 함수, QUALIFY, ROLLUP,
# STRING_AGG 등)을 거의 그대로 이해하고, pandas DataFrame을 이름으로 바로 조회할 수
# 있습니다. 별도의 "데이터 로드" 단계가 필요 없습니다.
import duckdb
import pandas as pd
from IPython.display import display

def run(sql: str) -> pd.DataFrame:
    """Execute a SQL string against DuckDB and return the result as a DataFrame.
    SQL 문자열을 DuckDB에서 실행하고 결과를 DataFrame으로 반환합니다."""
    return duckdb.sql(sql).df()

sales = pd.DataFrame({
    "sales_id": [1, 2, 3, 4, 5],
    "region":   ["서울", "서울", "부산", "부산", "서울"],
    "amount":   [45000, 61000, 32000, 95000, 28000],
})

print("-- GROUP BY: 5 rows -> collapses down to 2 rows (one per region) --")
print("-- GROUP BY: 5행 -> 2행으로 압축 (지역당 하나) --")
display(run("SELECT region, SUM(amount) AS region_total FROM sales GROUP BY region"))

print("-- Window function: still 5 rows -- each one now carries its region's total alongside it --")
print("-- 윈도우 함수: 여전히 5행 -- 각 행에 자기 지역의 합계가 함께 붙음 --")
sql = """
SELECT
    sales_id,
    region,
    amount,
    SUM(amount) OVER (PARTITION BY region) AS region_total
FROM sales
"""
display(run(sql))
# pandas equivalent: df.groupby("region")["amount"].transform("sum")


-- GROUP BY: 5 rows -> collapses down to 2 rows (one per region) --
-- GROUP BY: 5행 -> 2행으로 압축 (지역당 하나) --


,region,region_total
0,부산,127000.0
1,서울,134000.0


-- Window function: still 5 rows -- each one now carries its region's total alongside it --
-- 윈도우 함수: 여전히 5행 -- 각 행에 자기 지역의 합계가 함께 붙음 --


,sales_id,region,amount,region_total
0,1,서울,45000,134000.0
1,2,서울,61000,134000.0
2,5,서울,28000,134000.0
3,3,부산,32000,127000.0
4,4,부산,95000,127000.0


## Common Variations

In [2]:
# The general shape of any window function:
#   function() OVER (
#       PARTITION BY col   -- optional: which rows form a group ("GROUP BY", but nothing collapses)
#       ORDER BY col        -- optional: what order to process rows in (needed for LAG/RANK/running totals)
#   )
# 윈도우 함수의 일반적인 형태:
#   함수() OVER (
#       PARTITION BY 열   -- 선택: 어떤 행들이 한 그룹인지 ("GROUP BY"와 비슷하지만 압축 안 됨)
#       ORDER BY 열        -- 선택: 어떤 순서로 처리할지 (LAG/RANK/누적합에는 필수)
#   )

# Omitting both PARTITION BY and ORDER BY treats the entire table as a single window:
# PARTITION BY와 ORDER BY를 둘 다 생략하면 테이블 전체를 하나의 창으로 취급:
display(run("SELECT sales_id, region, amount, SUM(amount) OVER () AS grand_total FROM sales"))


,sales_id,region,amount,grand_total
0,1,서울,45000,261000.0
1,2,서울,61000,261000.0
2,3,부산,32000,261000.0
3,4,부산,95000,261000.0
4,5,서울,28000,261000.0


---
# 🧪 Small Examples

## Example 1 — ROW_NUMBER / RANK / DENSE_RANK: Ranking Functions / 순위 함수
**EN:** All three assign a rank based on `ORDER BY`, and they only differ in how they handle **ties**: `ROW_NUMBER` always gives distinct numbers even to tied rows (arbitrarily breaking the tie); `RANK` gives tied rows the same number, then **skips** the next rank(s) (two rows tied for 2nd → next is 4th); `DENSE_RANK` gives tied rows the same number too, but **doesn't skip** (two rows tied for 2nd → next is 3rd).  
**KR:** 세 함수 모두 `ORDER BY` 기준으로 순위를 매기며, **동점** 처리 방식만 다릅니다: `ROW_NUMBER`는 동점이어도 항상 서로 다른 번호를 줍니다(임의로 순서를 정함), `RANK`는 동점에 같은 번호를 주고 다음 순위를 **건너뜁니다**(2위 동점 2명 → 다음은 4위), `DENSE_RANK`는 동점에 같은 번호를 주지만 **건너뛰지 않습니다**(2위 동점 2명 → 다음은 3위).

In [3]:
monthly_sales = pd.DataFrame({
    "sales_id": [1, 2, 3, 4, 5],
    "name":     ["김민수", "이영희", "박준호", "최서연", "정대현"],
    "score":    [85, 92, 90, 78, 90],   # 박준호 and 정대현 are tied at 90 / 박준호·정대현 90점 동점
})

sql = """
SELECT
    name,
    score,
    ROW_NUMBER() OVER (ORDER BY score DESC) AS row_num,
    RANK()       OVER (ORDER BY score DESC) AS rnk,
    DENSE_RANK() OVER (ORDER BY score DESC) AS dense_rnk
FROM monthly_sales
"""
display(run(sql))
# 박준호/정대현 tie at 90: row_num gives them 2 and 3 (arbitrary), rnk gives them both 2 (then 김민수 jumps to 4),
# dense_rnk gives them both 2 (then 김민수 is 3, no gap).
# 박준호·정대현 90점 동점: row_num은 2,3(임의), rnk는 둘 다 2(다음 김민수는 4로 건너뜀),
# dense_rnk는 둘 다 2(다음 김민수는 구멍 없이 3).

print()
print("-- PARTITION BY + ORDER BY together: rank resets within each region --")
print("-- PARTITION BY + ORDER BY 함께: 각 지역 안에서 순위가 다시 시작 --")
sales1 = pd.DataFrame({
    "name":   ["김민수", "이영희", "박준호", "최서연", "정대현"],
    "region": ["서울", "서울", "부산", "부산", "서울"],
    "amount": [45000, 61000, 32000, 95000, 28000],
})
display(run("SELECT name, region, amount, RANK() OVER (PARTITION BY region ORDER BY amount DESC) AS rank_in_region FROM sales1"))
# pandas equivalent: df.groupby("region")["amount"].rank(ascending=False, method="min").astype(int)


,name,score,row_num,rnk,dense_rnk
0,이영희,92,1,1,1
1,박준호,90,2,2,2
2,정대현,90,3,2,2
3,김민수,85,4,4,3
4,최서연,78,5,5,4



-- PARTITION BY + ORDER BY together: rank resets within each region --
-- PARTITION BY + ORDER BY 함께: 각 지역 안에서 순위가 다시 시작 --


,name,region,amount,rank_in_region
0,최서연,부산,95000,1
1,박준호,부산,32000,2
2,이영희,서울,61000,1
3,김민수,서울,45000,2
4,정대현,서울,28000,3


## Example 2 — LAG / LEAD: Previous/Next Row Reference / 이전·다음 행 참조
**EN:** `LAG(col)` pulls in the value of `col` from the row *before* the current one (in `ORDER BY` order); `LEAD(col)` pulls from the row *after*. The first row has no `LAG`, and the last row has no `LEAD` — both come back `NULL`. This is the foundation of any "vs. previous period" calculation.  
**KR:** `LAG(열)`은 현재 행 *이전* 행(`ORDER BY` 순서 기준)의 값을 가져오고, `LEAD(열)`은 *다음* 행에서 가져옵니다. 첫 행은 `LAG`가 없고, 마지막 행은 `LEAD`가 없어서 둘 다 `NULL`이 됩니다. 모든 "전기 대비" 계산의 기초입니다.

In [4]:
monthly_revenue = pd.DataFrame({
    "month":   ["2024-01", "2024-02", "2024-03", "2024-04", "2024-05"],
    "revenue": [5200000, 4800000, 6100000, 5500000, 4900000],
})

display(run("""
SELECT month, revenue,
    LAG(revenue) OVER (ORDER BY month) AS prev_month_revenue,
    LEAD(revenue) OVER (ORDER BY month) AS next_month_revenue
FROM monthly_revenue
"""))

print("-- the real payoff: month-over-month % change --")
print("-- 진짜 활용: 전월 대비 변화율(%) --")
duckdb.sql("CREATE OR REPLACE MACRO safe_divide(a, b) AS CASE WHEN b = 0 THEN NULL ELSE a / b END")
sql = """
SELECT
    month,
    revenue,
    LAG(revenue) OVER (ORDER BY month) AS prev_revenue,
    revenue - LAG(revenue) OVER (ORDER BY month) AS mom_diff,
    ROUND(SAFE_DIVIDE(
        revenue - LAG(revenue) OVER (ORDER BY month),
        LAG(revenue) OVER (ORDER BY month)
    ) * 100, 1) AS mom_pct
FROM monthly_revenue
"""
display(run(sql))
# pandas equivalent: LAG = df["revenue"].shift(1)  |  MoM% = df["revenue"].pct_change() * 100


,month,revenue,prev_month_revenue,next_month_revenue
0,2024-01,5200000,<NA>,4800000
1,2024-02,4800000,5200000,6100000
2,2024-03,6100000,4800000,5500000
3,2024-04,5500000,6100000,4900000
4,2024-05,4900000,5500000,<NA>


-- the real payoff: month-over-month % change --
-- 진짜 활용: 전월 대비 변화율(%) --


,month,revenue,prev_revenue,mom_diff,mom_pct
0,2024-01,5200000,<NA>,<NA>,NaN
1,2024-02,4800000,5200000,-400000,-7.7
2,2024-03,6100000,4800000,1300000,27.1
3,2024-04,5500000,6100000,-600000,-9.8
4,2024-05,4900000,5500000,-600000,-10.9


## Example 3 — SUM OVER / AVG OVER: Running Totals & Moving Averages / 누적·이동 집계
**EN:** `ROWS BETWEEN ... AND ...` defines exactly which rows the window covers, relative to the current one. `UNBOUNDED PRECEDING AND CURRENT ROW` means "everything from the start up to here" — a running total. `2 PRECEDING AND CURRENT ROW` means "this row and the 2 before it" — a 3-period moving average.  
**KR:** `ROWS BETWEEN ... AND ...`는 현재 행 기준으로 창이 정확히 어떤 행들을 포함하는지 정의합니다. `UNBOUNDED PRECEDING AND CURRENT ROW`는 "처음부터 여기까지 전부"를 뜻하며 누적 합계가 됩니다. `2 PRECEDING AND CURRENT ROW`는 "이 행과 이전 2개 행"을 뜻하며 3개 구간 이동 평균이 됩니다.

In [5]:
print("-- running total / 누적합 --")
sql_cumsum = """
SELECT month, revenue,
    SUM(revenue) OVER (
        ORDER BY month
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_sum
FROM monthly_revenue
"""
display(run(sql_cumsum))

print("-- 3-period moving average / 3개월 이동 평균 --")
sql_ma = """
SELECT month, revenue,
    ROUND(AVG(revenue) OVER (
        ORDER BY month
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 0) AS ma3
FROM monthly_revenue
"""
display(run(sql_ma))
# early rows average over whatever's available (Jan: just itself: Feb: Jan+Feb) -- not an error, just fewer inputs.
# 초반 행은 있는 것만으로 평균 냄(1월: 자기 자신만, 2월: 1월+2월) -- 오류 아니고 그냥 입력이 적은 것.
# pandas equivalent: cumsum = df["revenue"].cumsum()  |  ma3 = df["revenue"].rolling(3, min_periods=1).mean()


-- running total / 누적합 --


,month,revenue,cumulative_sum
0,2024-01,5200000,5200000.0
1,2024-02,4800000,10000000.0
2,2024-03,6100000,16100000.0
3,2024-04,5500000,21600000.0
4,2024-05,4900000,26500000.0


-- 3-period moving average / 3개월 이동 평균 --


,month,revenue,ma3
0,2024-01,5200000,5200000.0
1,2024-02,4800000,5000000.0
2,2024-03,6100000,5366667.0
3,2024-04,5500000,5466667.0
4,2024-05,4900000,5500000.0


## Example 4 — FIRST_VALUE / LAST_VALUE: First & Last in a Window / 첫·마지막 값
**EN:** `FIRST_VALUE(col)` returns `col`'s value from the first row of the window (by `ORDER BY`); `LAST_VALUE(col)` returns it from the last row.

⚠️ **EN:** `LAST_VALUE` has a well-known trap: its **default frame is "up to the current row,"** which means without an explicit `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`, "the last row" is defined as *the current row itself* — so `LAST_VALUE` silently just returns each row's own value, not the window's true last value. Always spell out the full frame when using `LAST_VALUE`.  
⚠️ **KR:** `LAST_VALUE`에는 잘 알려진 함정이 있습니다: **기본 프레임(범위)이 "현재 행까지"**라서, 명시적으로 `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`을 쓰지 않으면 "마지막 행"이 *현재 행 자신*으로 정의됩니다 — 그래서 `LAST_VALUE`가 조용히 각 행 자신의 값만 반환하고, 창의 진짜 마지막 값을 주지 않습니다. `LAST_VALUE`를 쓸 때는 항상 전체 프레임을 명시하세요.

In [6]:
sales4 = pd.DataFrame({
    "sales_id": [1, 2, 3, 4, 5],
    "region":   ["서울", "서울", "서울", "부산", "부산"],
    "amount":   [45000, 61000, 28000, 32000, 95000],
})

sql = """
SELECT
    sales_id, region, amount,
    FIRST_VALUE(amount) OVER (PARTITION BY region ORDER BY amount DESC) AS max_in_region,
    LAST_VALUE(amount) OVER (
        PARTITION BY region ORDER BY amount DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS min_in_region
FROM sales4
"""
display(run(sql))

print("-- ⚠️ THE TRAP: LAST_VALUE WITHOUT the explicit frame --")
print("-- ⚠️ 함정: 명시적 프레임 없는 LAST_VALUE --")
sql_trap = """
SELECT sales_id, region, amount,
    LAST_VALUE(amount) OVER (PARTITION BY region ORDER BY amount DESC) AS last_value_broken
FROM sales4
"""
display(run(sql_trap))
# Notice last_value_broken == amount on every single row -- useless. That's the default-frame trap.
# last_value_broken이 모든 행에서 amount와 똑같음을 주목 -- 쓸모없음. 이게 기본 프레임 함정.


,sales_id,region,amount,max_in_region,min_in_region
0,5,부산,95000,95000,32000
1,4,부산,32000,95000,32000
2,2,서울,61000,61000,28000
3,1,서울,45000,61000,28000
4,3,서울,28000,61000,28000


-- ⚠️ THE TRAP: LAST_VALUE WITHOUT the explicit frame --
-- ⚠️ 함정: 명시적 프레임 없는 LAST_VALUE --


,sales_id,region,amount,last_value_broken
0,5,부산,95000,95000
1,4,부산,32000,32000
2,2,서울,61000,61000
3,1,서울,45000,45000
4,3,서울,28000,28000


## Example 5 — NTILE: Splitting Rows Into Equal Buckets / 분위수 버킷 나누기
**EN:** `NTILE(n)` divides the rows (in `ORDER BY` order) into `n` roughly-equal-sized buckets, numbered 1 through `n`. `NTILE(4)` gives quartiles — `quartile = 1` is the top 25% (if sorted descending), `quartile = 4` is the bottom 25%. If the row count doesn't divide evenly, the earlier buckets get the extra row(s).  
**KR:** `NTILE(n)`은 (`ORDER BY` 순서로) 행을 대략 같은 크기의 `n`개 버킷으로 나누고 1부터 `n`까지 번호를 매깁니다. `NTILE(4)`는 4분위수를 만들며 — 내림차순 정렬 시 `quartile = 1`이 상위 25%, `quartile = 4`가 하위 25%입니다. 행 수가 딱 나누어떨어지지 않으면 앞쪽 버킷에 여분의 행이 배분됩니다.

In [7]:
customers5 = pd.DataFrame({
    "customer_id": ["C01","C02","C03","C04","C05","C06","C07","C08"],
    "name":        ["김민수","이영희","박준호","최서연","정대현","윤소영","한지훈","강다은"],
    "total_spent": [1200000, 450000, 85000, 2100000, 320000, 680000, 95000, 1750000],
})

sql = """
SELECT name, total_spent, NTILE(4) OVER (ORDER BY total_spent DESC) AS quartile
FROM customers5
"""
display(run(sql))
# 8 customers / 4 buckets = 2 per bucket, evenly.
# pandas equivalent: pd.qcut(df["total_spent"], q=4, labels=["Q4","Q3","Q2","Q1"]) -- qcut splits by
# frequency, NTILE splits by sorted position -- subtly different when there are ties.


,name,total_spent,quartile
0,최서연,2100000,1
1,강다은,1750000,1
2,김민수,1200000,2
3,윤소영,680000,2
4,이영희,450000,3
5,정대현,320000,3
6,한지훈,95000,4
7,박준호,85000,4


## Example 6 — QUALIFY: Filtering on a Window Function's Result / 윈도우 함수 결과 필터링
**EN:** You can't put a window function directly in `WHERE` (window functions run *after* `WHERE`, in the execution order). `QUALIFY` exists specifically to solve this — it filters on a window function's result the same way `HAVING` filters on an aggregate's result, without needing to wrap the query in a subquery.  
**KR:** 윈도우 함수는 `WHERE`에 직접 쓸 수 없습니다(윈도우 함수는 실행 순서상 `WHERE` *이후에* 실행됩니다). `QUALIFY`는 정확히 이 문제를 풀기 위해 존재합니다 — `HAVING`이 집계 결과를 필터링하는 것과 같은 방식으로 윈도우 함수 결과를 필터링하며, 쿼리를 서브쿼리로 감쌀 필요가 없습니다.

⚠️ `QUALIFY` is a **BigQuery/Snowflake-specific** extension — PostgreSQL and MySQL don't support it, and require the subquery form shown below instead.  
⚠️ `QUALIFY`는 **BigQuery/Snowflake 전용** 확장 문법입니다 — PostgreSQL과 MySQL은 지원하지 않으며, 대신 아래에 나온 서브쿼리 형태가 필요합니다.

In [8]:
orders6 = pd.DataFrame({
    "order_id":    [1001, 1002, 1003, 1004, 1005],
    "customer_id": ["C01", "C01", "C02", "C02", "C02"],
    "order_date":  ["2024-01-15", "2024-02-03", "2024-01-22", "2024-03-10", "2024-03-28"],
    "amount":      [45000, 61000, 32000, 95000, 28000],
})

print("-- subquery version: works on any SQL engine, but needs an extra layer --")
print("-- 서브쿼리 버전: 어느 SQL 엔진에서도 동작하지만 한 겹 더 필요 --")
sql_sub = """
SELECT * FROM (
    SELECT *,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC) AS rn
    FROM orders6
)
WHERE rn = 1
"""
display(run(sql_sub))

print("-- QUALIFY version: same result, no extra layer needed --")
print("-- QUALIFY 버전: 결과는 같지만 겹칠 필요 없음 --")
sql_qualify = """
SELECT *
FROM orders6
QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC) = 1
"""
display(run(sql_qualify))
# both return each customer's single most recent order.
# 둘 다 각 고객의 가장 최근 주문 하나씩만 반환.


-- subquery version: works on any SQL engine, but needs an extra layer --
-- 서브쿼리 버전: 어느 SQL 엔진에서도 동작하지만 한 겹 더 필요 --


,order_id,customer_id,order_date,amount,rn
0,1002,C01,2024-02-03,61000,1
1,1005,C02,2024-03-28,28000,1


-- QUALIFY version: same result, no extra layer needed --
-- QUALIFY 버전: 결과는 같지만 겹칠 필요 없음 --


,order_id,customer_id,order_date,amount
0,1002,C01,2024-02-03,61000
1,1005,C02,2024-03-28,28000


## Example 7 — Common Combinations / 자주 쓰는 조합
**EN:** **Pattern A** combines `RANK` with `QUALIFY` for the classic "top N per category" report — no subquery needed. **Pattern B** combines `LAG` with `CASE WHEN` (Chapter 5) to turn a raw percentage change into a readable trend arrow — a small touch that makes a table much easier for a stakeholder to scan.
**KR:** **패턴 A**는 `RANK`와 `QUALIFY`를 조합해 고전적인 "카테고리별 상위 N개" 리포트를 서브쿼리 없이 만듭니다. **패턴 B**는 `LAG`와 (5장의) `CASE WHEN`을 조합해 단순 변화율 숫자를 읽기 쉬운 추세 화살표로 바꿉니다 — 이해관계자가 표를 훑어보기 훨씬 쉽게 만드는 작은 손질입니다.

In [9]:
print("-- Pattern A: RANK + QUALIFY -- top 2 products per category --")
print("-- 패턴 A: RANK + QUALIFY -- 카테고리별 상위 2개 제품 --")
products = pd.DataFrame({
    "product":  ["노트북 Pro", "스마트폰", "이어폰", "재킷", "청바지", "티셔츠"],
    "category": ["전자", "전자", "전자", "의류", "의류", "의류"],
    "revenue":  [3600000, 2800000, 950000, 1200000, 850000, 430000],
})
sql_a = """
SELECT product, category, revenue
FROM products
QUALIFY RANK() OVER (PARTITION BY category ORDER BY revenue DESC) <= 2
"""
display(run(sql_a))

print("-- Pattern B: LAG + CASE WHEN -- trend arrows --")
print("-- 패턴 B: LAG + CASE WHEN -- 증감 방향 화살표 --")
mr7 = pd.DataFrame({"month": ["2024-01","2024-02","2024-03","2024-04"], "revenue": [5200000,4800000,6100000,5500000]})
sql_b = """
SELECT
    month,
    revenue,
    LAG(revenue) OVER (ORDER BY month) AS prev,
    ROUND(SAFE_DIVIDE(
        revenue - LAG(revenue) OVER (ORDER BY month),
        LAG(revenue) OVER (ORDER BY month)
    ) * 100, 1) AS mom_pct,
    CASE
        WHEN revenue > LAG(revenue) OVER (ORDER BY month) THEN '▲ 증가'
        WHEN revenue < LAG(revenue) OVER (ORDER BY month) THEN '▼ 감소'
        ELSE '→ 유지'
    END AS trend
FROM mr7
"""
display(run(sql_b))


-- Pattern A: RANK + QUALIFY -- top 2 products per category --
-- 패턴 A: RANK + QUALIFY -- 카테고리별 상위 2개 제품 --


,product,category,revenue
0,노트북 Pro,전자,3600000
1,스마트폰,전자,2800000
2,재킷,의류,1200000
3,청바지,의류,850000


-- Pattern B: LAG + CASE WHEN -- trend arrows --
-- 패턴 B: LAG + CASE WHEN -- 증감 방향 화살표 --


,month,revenue,prev,mom_pct,trend
0,2024-01,5200000,<NA>,NaN,→ 유지
1,2024-02,4800000,5200000,-7.7,▼ 감소
2,2024-03,6100000,4800000,27.1,▲ 증가
3,2024-04,5500000,6100000,-9.8,▼ 감소


## Example 8 — Practice / 실습 문제
**EN:** Fill in each `________` blank below, then remove the `#` in front of the matching `display(run(...))` line to check your answer. Hints: `DENSE_RANK` `region` `UNBOUNDED PRECEDING` `CURRENT ROW` `DESC`
**KR:** 아래 `________` 빈칸을 채운 뒤, 해당 `display(run(...))` 줄 앞의 `#`을 지우고 실행해서 답을 확인하세요. 힌트: `DENSE_RANK` `region` `UNBOUNDED PRECEDING` `CURRENT ROW` `DESC`

In [12]:
regional_sales = pd.DataFrame({
    "sales_id":    [1, 2, 3, 4, 5, 6],
    "region":      ["서울", "서울", "서울", "부산", "부산", "부산"],
    "salesperson": ["김민수", "이영희", "박준호", "최서연", "정대현", "윤소영"],
    "amount":      [320000, 450000, 450000, 210000, 380000, 160000],
    "sale_date":   ["2024-01-10", "2024-01-22", "2024-02-05", "2024-01-15", "2024-02-01", "2024-02-20"],
})

# Q1. Rank sales within each region by amount (use DENSE_RANK -- ties should NOT skip a rank).
# Q1. 지역 안에서 sales 금액 순위(DENSE_RANK, 동점이면 다음 순위를 건너뛰지 않음).
q1 = """
SELECT
    salesperson,
    region,
    amount,
    DENSE_RANK() OVER (PARTITION BY region ORDER BY amount DESC) AS rank_in_region
FROM regional_sales
"""
display(run(q1))   # <- uncomment once filled in / 빈칸을 채운 뒤 주석 해제

# Q2. Running total of amount, ordered by date, across the whole table (ignore region).
# Q2. 날짜 순서로 전체 기준 누적 매출(SUM OVER).
q2 = """
SELECT
    sale_date,
    salesperson,
    amount,
    SUM(amount) OVER (
        ORDER BY sale_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative
FROM regional_sales
"""
display(run(q2))   # <- uncomment once filled in / 빈칸을 채운 뒤 주석 해제

# Q3. Pull only each region's single most recent transaction (use QUALIFY).
# Q3. 지역별로 가장 최근 거래만 뽑아라 (QUALIFY 사용).
q3 = """
SELECT salesperson, region, amount, sale_date
FROM regional_sales
QUALIFY
    ROW_NUMBER() OVER (PARTITION BY region ORDER BY sale_date DESC) = 1
"""
display(run(q3))   # <- uncomment once filled in / 빈칸을 채운 뒤 주석 해제

print("✏️  Fill in the ________ blanks above, uncomment the display() lines, then re-run this cell.")
print("✏️  위 ________ 빈칸을 채우고 display() 줄의 주석을 해제한 뒤 이 셀을 다시 실행하세요.")


,salesperson,region,amount,rank_in_region
0,이영희,서울,450000,1
1,박준호,서울,450000,1
2,김민수,서울,320000,2
3,정대현,부산,380000,1
4,최서연,부산,210000,2
5,윤소영,부산,160000,3


,sale_date,salesperson,amount,cumulative
0,2024-01-10,김민수,320000,320000.0
1,2024-01-15,최서연,210000,530000.0
2,2024-01-22,이영희,450000,980000.0
3,2024-02-01,정대현,380000,1360000.0
4,2024-02-05,박준호,450000,1810000.0
5,2024-02-20,윤소영,160000,1970000.0


,salesperson,region,amount,sale_date
0,박준호,서울,450000,2024-02-05
1,윤소영,부산,160000,2024-02-20


✏️  Fill in the ________ blanks above, uncomment the display() lines, then re-run this cell.
✏️  위 ________ 빈칸을 채우고 display() 줄의 주석을 해제한 뒤 이 셀을 다시 실행하세요.


<details>
<summary>🔑 Answer / 정답 (click to expand / 클릭해서 펼치기)</summary>

```sql
-- Q1
SELECT
    salesperson,
    region,
    amount,
    DENSE_RANK() OVER (PARTITION BY region ORDER BY amount DESC) AS rank_in_region
FROM regional_sales

-- Q2
SELECT
    sale_date,
    salesperson,
    amount,
    SUM(amount) OVER (
        ORDER BY sale_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative
FROM regional_sales

-- Q3
SELECT salesperson, region, amount, sale_date
FROM regional_sales
QUALIFY
    ROW_NUMBER() OVER (PARTITION BY region ORDER BY sale_date DESC) = 1
```
</details>

---
# ⚠️ Common Mistakes

**Mistake 1 — Forgetting `ORDER BY` inside `OVER()` for `LAG`/`RANK`/running totals**
- EN: `LAG`, `RANK`, and running-total `SUM() OVER (...)` all fundamentally depend on a defined *order* — without `ORDER BY` inside the `OVER()`, "the previous row" or "the running total so far" isn't a meaningful concept, and results become unpredictable.
- KR: `LAG`, `RANK`, 누적합 `SUM() OVER (...)`는 모두 근본적으로 정의된 *순서*에 의존합니다 — `OVER()` 안에 `ORDER BY`가 없으면 "이전 행"이나 "여기까지의 누적합"이라는 개념 자체가 성립하지 않고, 결과가 예측 불가능해집니다.
- ✅ Fix / 해결법: Always include `ORDER BY` inside `OVER()` for any function where "order" is part of the meaning.  
"순서"가 의미의 일부인 함수라면 `OVER()` 안에 항상 `ORDER BY`를 포함하세요.

**Mistake 2 — Using `LAST_VALUE` without the full frame**
- EN: As Example 4 showed, `LAST_VALUE`'s default frame stops at the current row, so it silently returns each row's own value instead of the window's actual last value — no error, just a quietly useless column.
- KR: 예제 4에서 봤듯이, `LAST_VALUE`의 기본 프레임은 현재 행에서 멈추므로 창의 진짜 마지막 값 대신 조용히 각 행 자신의 값을 반환합니다 — 오류는 없고, 그냥 조용히 쓸모없는 열이 됩니다.
- ✅ Fix / 해결법: Always add `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` when using `LAST_VALUE` — or better, use `FIRST_VALUE` with the sort order reversed, which doesn't have this trap.  
`LAST_VALUE`를 쓸 때는 항상 `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`을 추가하세요 — 더 좋게는, 이 함정이 없는 `FIRST_VALUE`를 정렬 순서를 뒤집어서 대신 쓰세요.

**Mistake 3 — Putting a window function directly in `WHERE`**
- EN: `WHERE ROW_NUMBER() OVER (...) = 1` throws an error — window functions execute *after* `WHERE` in SQL's real order, so `WHERE` can't see their result yet.
- KR: `WHERE ROW_NUMBER() OVER (...) = 1`은 오류가 납니다 — 윈도우 함수는 SQL의 실제 실행 순서상 `WHERE` *이후에* 실행되므로, `WHERE`는 아직 그 결과를 볼 수 없습니다.
- ✅ Fix / 해결법: Use `QUALIFY` (BigQuery/Snowflake) or wrap the query in a subquery and filter the outer `WHERE` on the computed column (portable to any engine).  
`QUALIFY`(BigQuery/Snowflake)를 쓰거나, 쿼리를 서브쿼리로 감싸서 바깥 `WHERE`가 계산된 열을 필터링하게 하세요(어느 엔진에서나 이식 가능).

**Mistake 4 — Reaching for `RANK` when you actually need `DENSE_RANK` (or vice versa)**
- EN: `RANK` skips numbers after a tie (two rows tied for 1st → next row is 3rd); `DENSE_RANK` doesn't. If a report is meant to show "how many distinct performance tiers exist," `RANK`'s gaps will overstate the count of tiers.
- KR: `RANK`는 동점 뒤 번호를 건너뛰고(1위 동점 2명 → 다음은 3위), `DENSE_RANK`는 건너뛰지 않습니다. 리포트가 "서로 다른 성과 등급이 몇 개 있는지"를 보여주려는 것이라면, `RANK`의 건너뛰기가 등급 수를 실제보다 많아 보이게 만듭니다.
- ✅ Fix / 해결법: Use `RANK` when gaps should reflect "how many rows beat this one"; use `DENSE_RANK` when you want a clean, gap-free tier count.  
"이 행보다 앞선 행이 몇 개인지"를 반영해야 하면 `RANK`, 구멍 없는 깔끔한 등급 수가 필요하면 `DENSE_RANK`를 쓰세요.

---
# 💡 Tips
Useful tips or shortcuts / 유용한 팁과 단축법

- When stuck, ask "does this need to collapse rows, or keep them all?" — collapse → `GROUP BY`; keep them all → a window function. This single question resolves most GROUP-BY-vs-window confusion.  
 막힐 때는 "행을 압축해야 하나, 다 유지해야 하나?"를 물어보세요 — 압축 → `GROUP BY`, 다 유지 → 윈도우 함수. 이 질문 하나로 GROUP BY vs 윈도우 함수 혼란의 대부분이 풀립니다.
- `ROW_NUMBER() = 1` after a `PARTITION BY` + `ORDER BY` is the single most common window-function pattern in real work — "get the latest/largest/first row per group" — memorize this shape as one unit.  
 `PARTITION BY` + `ORDER BY` 뒤에 오는 `ROW_NUMBER() = 1`은 실무에서 가장 흔한 윈도우 함수 패턴입니다 — "그룹별 최신·최대·첫 행 가져오기" — 이 형태를 하나의 단위로 외워두세요.
- `QUALIFY` removes a whole layer of subquery nesting — once your engine supports it, you'll rarely go back to the subquery form for this kind of filtering.  
 `QUALIFY`는 서브쿼리 중첩 한 겹을 통째로 없애줍니다 — 엔진이 지원하기 시작하면 이런 종류의 필터링에 서브쿼리 형태로 돌아갈 일이 거의 없어집니다.
- If a window function result looks suspiciously uniform (like every row having the same value), suspect the frame first — it's very often the `LAST_VALUE` trap from Mistake 2.  
 윈도우 함수 결과가 의심스러울 정도로 균일해 보인다면(모든 행이 같은 값처럼) 먼저 프레임을 의심하세요 — 실수 2번의 `LAST_VALUE` 함정인 경우가 매우 흔합니다.

---
# 🔗 Related Concepts

```
SQL Learning Roadmap (this guide) / SQL 학습 로드맵 (이 가이드)
──────────────────────────────────────────────
 1. SELECT Basics
 2. Aggregation & GROUP BY
 3. JOIN
 4. Subquery & CTE
 5. Conditions & NULL Handling
 6. String & Date Functions
 7. Window Functions             ← ★ YOU ARE HERE / 지금 여기
 8. BA-Specific Patterns
```

```
GROUP BY vs. window function, side by side / GROUP BY vs 윈도우 함수 나란히 비교
──────────────────────────────────────────────
  Input: 5 rows                       Input: 5 rows
  GROUP BY region                     SUM(amount) OVER (PARTITION BY region)
        │                                    │
        ▼                                    ▼
  Output: 2 rows (collapsed)          Output: 5 rows (all kept)
  출력: 2행 (압축됨)                     출력: 5행 (전부 유지)
```

*How is today's topic connected to other concepts?*

**EN:** Window functions build directly on Chapter 2's aggregate functions (`SUM`, `COUNT`, `AVG` all work equally well inside `OVER (...)`) and Chapter 5's `CASE WHEN` (Pattern B turns a window result into a readable label). `QUALIFY` mirrors Chapter 2's `HAVING` almost exactly — same job, one level up the execution order. Looking ahead, Chapter 8 (BA-specific patterns) is essentially a showcase of window functions applied to real business questions — MoM/YoY growth, running totals, and top-N-per-group are all this chapter's tools, just given business names.

**KR:** 윈도우 함수는 2장의 집계 함수(`SUM`, `COUNT`, `AVG` 모두 `OVER (...)` 안에서 똑같이 잘 동작함) 위에 직접 세워지며, 5장의 `CASE WHEN`(패턴 B가 윈도우 결과를 읽기 쉬운 라벨로 바꿈)도 활용합니다. `QUALIFY`는 2장의 `HAVING`과 거의 똑같은 역할을 실행 순서 한 단계 위에서 합니다. 앞으로 배울 8장(BA 특화 패턴)은 사실상 윈도우 함수를 실제 비즈니스 질문에 적용하는 전시장입니다 — MoM/YoY 성장률, 누적 합계, 그룹별 상위 N개 모두 이번 챕터의 도구에 비즈니스 이름을 붙인 것뿐입니다.

---
# 💼 Business Example
*How would a Business Analyst use this?*

**Scenario / 시나리오:**
**EN:** Sales leadership asks: *"For each region, who's the top performer, and how do we know if they're actually #1 or tied with someone?"* This needs `RANK` (not `ROW_NUMBER`, since ties matter here) combined with `QUALIFY` to isolate just the leaders.
**KR:** 영업 리더십이 묻습니다: *"각 지역에서 최고 실적자가 누구고, 정말 단독 1위인지 아니면 동점인지 어떻게 알 수 있어?"* 여기서는 동점이 중요하므로 `ROW_NUMBER`가 아니라 `RANK`가 필요하고, `QUALIFY`로 1위만 골라냅니다.

**To-do / 할 일:**
- [x] Rank salespeople within each region by amount, using `RANK` so ties show correctly  
`RANK`로 지역 안에서 담당자 순위를 매겨 동점이 올바르게 보이게 한다
- [x] Keep only rank 1 with `QUALIFY`  
`QUALIFY`로 1위만 남긴다
- [x] Present region, salesperson, and amount together  
지역, 담당자, 금액을 함께 보여준다

In [11]:
sales_biz = pd.DataFrame({
    "region":      ["서울", "서울", "서울", "부산", "부산", "부산"],
    "salesperson": ["김민수", "이영희", "박준호", "최서연", "정대현", "윤소영"],
    "amount":      [450000, 450000, 320000, 380000, 210000, 160000],
})

sql = """
SELECT region, salesperson, amount
FROM sales_biz
QUALIFY RANK() OVER (PARTITION BY region ORDER BY amount DESC) = 1
"""
display(run(sql))
# 서울 shows TWO names -- 김민수 and 이영희 are genuinely tied for #1, and RANK (not ROW_NUMBER) surfaces that honestly.
# 서울에서 이름이 두 개 나옴 -- 김민수와 이영희가 진짜 1위 동점이며, ROW_NUMBER가 아니라 RANK를 썼기 때문에 이걸 정직하게 보여줌.


,region,salesperson,amount
0,부산,최서연,380000
1,서울,김민수,450000
2,서울,이영희,450000


---
# 📝 Summary
*Write today's concept in 3~5 sentences.*

**EN:** A window function computes a value across a group of rows *without collapsing them*, unlike `GROUP BY` — every original row survives, with a calculated value attached via `OVER (PARTITION BY ... ORDER BY ...)`. `ROW_NUMBER`/`RANK`/`DENSE_RANK` differ only in tie-handling; `LAG`/`LEAD` reference a neighboring row, powering any "vs. previous period" calculation; `SUM`/`AVG OVER (... ROWS BETWEEN ...)` build running totals and moving averages by defining an explicit frame. `FIRST_VALUE`/`LAST_VALUE` pull an endpoint value from the window, though `LAST_VALUE` needs its frame spelled out in full to avoid silently returning the current row. `NTILE` splits ordered rows into equal buckets, and `QUALIFY` (BigQuery/Snowflake) filters directly on a window function's result, skipping the subquery layer other engines require.

**KR:** 윈도우 함수는 `GROUP BY`와 달리 행을 *압축하지 않고* 행 그룹에 걸쳐 값을 계산합니다 — 원본 행이 모두 살아남고, `OVER (PARTITION BY ... ORDER BY ...)`로 계산된 값이 각 행에 붙습니다. `ROW_NUMBER`/`RANK`/`DENSE_RANK`는 동점 처리 방식만 다르고, `LAG`/`LEAD`는 이웃 행을 참조해서 모든 "전기 대비" 계산의 동력이 됩니다. `SUM`/`AVG OVER (... ROWS BETWEEN ...)`는 명시적인 프레임을 정의해서 누적 합계와 이동 평균을 만듭니다. `FIRST_VALUE`/`LAST_VALUE`는 창의 끝 값을 가져오는데, `LAST_VALUE`는 조용히 현재 행 값만 반환하는 걸 피하려면 프레임을 전부 명시해야 합니다. `NTILE`은 정렬된 행을 균등한 버킷으로 나누고, `QUALIFY`(BigQuery/Snowflake)는 다른 엔진이 요구하는 서브쿼리 층을 건너뛰고 윈도우 함수 결과를 직접 필터링합니다.

---
# 📌 One Sentence Summary
Today's topic in ONE sentence. / 오늘 배운 내용을 한 문장으로.

> **EN:** A window function is `GROUP BY` that refuses to erase your rows — it attaches the group-level answer directly onto every individual row instead, which is exactly what makes ranking, running totals, and "compare to the previous row" possible in a single query.

> **KR:** 윈도우 함수는 행을 지우기를 거부하는 `GROUP BY`입니다 — 그룹 단위 답을 개별 행 각각에 그대로 붙여주며, 바로 이 점이 순위 매기기, 누적 합계, "이전 행과 비교하기"를 한 쿼리로 가능하게 만듭니다.

---
# ❓ Review Questions

**Q1.** `GROUP BY` and a window function can both compute a "total per region." What's the one fundamental difference in their output, and why does that difference matter?  
**Q1.** `GROUP BY`와 윈도우 함수 둘 다 "지역별 합계"를 계산할 수 있다. 둘의 출력에는 근본적으로 어떤 차이가 있으며, 그 차이가 왜 중요한가?

GROUP BY collapses rows to one per group; a window function keeps every row and attaches the group total as a new column.  
GROUP BY는 행을 지역당 1행으로 압축합니다. 윈도우 함수는 원본 행을 모두 남기고 각 행 옆에 지역 합계를 붙입니다. 그래서 "이 주문 금액 + 그 지역 총합"처럼 개별 값과 그룹 값을 한 줄에 나란히 볼 수 있습니다.

**Q2.** Three salespeople are tied for 2nd place. What rank does the 5th-place (non-tied) salesperson get under `RANK` vs. `DENSE_RANK`?  
**Q2.** 영업사원 세 명이 2위로 동점이다. 동점이 아닌 5등 영업사원은 `RANK`와 `DENSE_RANK`에서 각각 몇 등이 되는가?

With three tied at 2nd, the next non-tied person gets 5 under RANK and 3 under DENSE_RANK.  
RANK는 3·4를 건너뛰고 5등이 됩니다. DENSE_RANK는 다음 순위가 3등입니다.

**Q3.** You write `LAST_VALUE(amount) OVER (ORDER BY date)` and every row shows its own `amount` back, which seems pointless. What's the actual bug, and how do you fix it?  
**Q3.** `LAST_VALUE(amount) OVER (ORDER BY date)`를 썼더니 모든 행이 자기 자신의 `amount`를 그대로 보여줘서 의미가 없어 보인다. 실제 버그는 무엇이고 어떻게 고치는가?

The default frame stops at the current row, so "last" means the current row itself. Add ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING.  
LAST_VALUE의 기본 프레임은 "처음부터 현재 행까지"라서, "마지막"이 창 전체가 아니라 현재 행 자신이 됩니다. 고치려면 LAST_VALUE(amount) OVER (ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)처럼 전체 프레임을 명시해야 합니다.

**Q4.** Why does `WHERE ROW_NUMBER() OVER (...) = 1` fail, and what are the two different ways to fix it?  
**Q4.** 왜 `WHERE ROW_NUMBER() OVER (...) = 1`은 실패하며, 이를 고치는 서로 다른 두 가지 방법은 무엇인가?

윈도우 함수는 실행 순서상 WHERE 이후에 계산되므로, WHERE 안에서는 그 결과를 쓸 수 없습니다. 고치는 방법은 두 가지입니다. (1) QUALIFY ROW_NUMBER() OVER (...) = 1 — BigQuery/Snowflake용. (2) 서브쿼리로 순위 열을 만든 뒤, 바깥 WHERE rn = 1 — 다른 엔진에서도 동작합니다.
Window functions run after WHERE, so WHERE cannot filter on them. Fix with QUALIFY or a subquery plus an outer WHERE.

**Q5.** You want each product category's top 3 best-selling items in one query, without a subquery. Which two window-function tools from this chapter combine to do that, and how?  
**Q5.** 서브쿼리 없이 한 쿼리로 각 제품 카테고리의 베스트셀러 상위 3개를 구하고 싶다. 이번 챕터에서 배운 어떤 두 윈도우 함수 도구를 조합해야 하며, 어떻게 조합하는가?

Combine a ranking function with PARTITION BY category and filter with QUALIFY ... <= 3.  
순위 함수(RANK() 또는 ROW_NUMBER())와 **QUALIFY**를 조합합니다. 예: RANK() OVER (PARTITION BY category ORDER BY sales DESC)로 카테고리 안 순위를 매기고, QUALIFY RANK() OVER (PARTITION BY category ORDER BY sales DESC) <= 3으로 상위 3개만 남깁니다. PARTITION BY category로 카테고리마다 순위가 따로 매겨집니다.

---
*📅 Try answering these again in a few days. / 며칠 후 다시 답해보세요.*